# Realizability Test — Predicted (not observed) upstream Q

The load-bearing test: does the +0.037 upstream-flow gain survive when upstream discharge is **predicted** from forcings rather than observed? If yes, the method is **deployable** (no ground truth at inference) → real paper. If ~0, the +0.037 lives only as an upper bound.

Two-stage, no target leakage (pre-registered in `preregistration_realizability.md`):
1. Run the trained L baseline over the full span (1990–2008) → predicted Q per basin.
2. Train L + `upstream_q_pred` (area-weighted upstream *predicted* Q, lag 1) and compare to L (0.653) and the oracle L+upQ (0.703).

**Success ≥ +0.015 (recovers ≥40% of the +0.037 ceiling). Falsify ≤ +0.005.**

Runs on GPU because the full dataset lives on Drive (local Mac has a subset). T4 → Run all.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config

In [ ]:
import os
GITHUB_URL = 'https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH = ''
SEED = 11
AUTO = ['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
        '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; print('Found', c); break
    else: raise RuntimeError('set DRIVE_CAMELS_PATH')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydrology_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('seed', SEED)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'
import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import importlib,sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR,'runs','topology_ablation','component0'), exist_ok=True)
print('symlinks ready')

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Ensure the L baseline run exists on Drive

The realizability test reuses the trained L baseline. If `runs/topology_ablation/component0/L_component0_seed11/` isn't on Drive, train it (stock cudalstm, ~4 min on T4).

In [ ]:
%cd {REPO_DIR}
import glob
Ldir=f'{REPO_DIR}/runs/topology_ablation/component0/L_component0_seed11'
if not os.path.isfile(f'{Ldir}/model_epoch030.pt'):
    print('L baseline not found on Drive — training it...')
    !python experiments/topology_ablation/make_configs.py --network component0 \
        --basin-file topology_analysis/phase1_network_discovery/outputs/component0_basins.txt \
        --seed {SEED} --device cuda:0 --epochs 30
    !python neuralhydrology/nh_run.py train --config-file experiments/topology_ablation/configs/L_component0_seed{SEED}.yaml 2>&1 | tail -3
    ts=sorted(glob.glob(f'{REPO_DIR}/runs/topology_ablation/component0/L_component0_seed{SEED}_*'))
    if ts: os.rename(ts[-1], Ldir)
    !python neuralhydrology/nh_run.py evaluate --run-dir {Ldir} --epoch 30 2>&1 | tail -2
else:
    print('L baseline present:', Ldir)

## Cell 8 — Stage 1: build predicted-upstream-Q (full-span L predictions → upstream aggregate)

In [ ]:
%cd {REPO_DIR}
!python experiments/topology_ablation/build_predicted_upstream_q.py --network component0 --seed {SEED} --lag-days 1 2>&1 | tail -6

## Cell 9 — Stage 2: train L + predicted-upstream-Q, compare to L and the oracle

In [ ]:
%cd {REPO_DIR}
!python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 \
    --feature-file experiments/topology_ablation/features/upstream_q_pred_component0_seed{SEED}_lag1.p \
    --cond-name L_upQpred 2>&1 | tail -6

## Cell 10 — Verdict vs pre-registration

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, os
base=f'{REPO_DIR}/runs/topology_ablation/component0'
def nse(c):
    p=f'{base}/{c}_component0_seed{SEED}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.exists(p) else None
L=nse('L'); P=nse('L_upQpred'); O=nse('L_upQ')
print(f'L         median NSE: {L.median():+.4f}')
if O is not None: print(f'L+upQ     (oracle)  : {O.median():+.4f}  (+{(O.median()-L.median()):.3f})')
if P is not None:
    b=L.index.intersection(P.index); d=(P.loc[b]-L.loc[b]).dropna()
    g=d.median(); ceil=0.037
    print(f'L+upQpred (realizable): {P.median():+.4f}')
    print(f'\nupQpred - L paired: median {g:+.4f}  frac>0 {(d>0).mean():.2f}  n={len(d)}')
    print(f'Recovers {100*g/ceil:.0f}% of the +0.037 oracle ceiling')
    v = 'SUCCESS (deployable)' if g>=0.015 else ('PARTIAL' if g>=0.005 else 'FALSIFIED (not deployable)')
    print(f'PRE-REG VERDICT: {v}')

## Done

Pull the verdict + `runs/topology_ablation/component0/L_upQpred_*` to local, then `crs interpret realizability`. If SUCCESS or PARTIAL → multi-seed confirmation is next. If FALSIFIED → reframe paper around the upper bound.